# Pre-entrega: Persistencia y la integración de scripts ETL
**Curso:** Data Science II — Coderhouse  
**Módulo 9:** Automatización y Patrones de Datos (págs. 187 - 188)  
**Estudiante:** Paulo Martin  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulo78marti-droid/Pre-entrega-8--Persistencia-y-la-integraci-n-de-scripts-ETL/blob/master/Pre-entrega_8_ETL.ipynb)

---

## Propósito de la Entrega
El objetivo de esta actividad es implementar un flujo de trabajo automatizado que integre las tres etapas de un proceso **ETL (Extract, Transform, Load)** y persista los datos en una base de datos analítica local (**SQLite**) asegurando repetibilidad, manejo de errores y prevención de duplicados.

### Fases del Pipeline:
1. **Extracción (Extract):** Descarga del dataset de Calidad del Aire (`srisyra02/air-quality-dataset`) mediante la librería `kagglehub` y guardado en área de transición (*Staging Area*).
2. **Transformación (Transform):** Limpieza con Pandas, estandarización de columnas, tratamiento de valores negativos en sensores, *feature engineering* (categorización según estándares de PM2.5) y generación de una tabla de resumen analítico.
3. **Carga y Persistencia (Load):** Almacenamiento estructurado en SQLite con **SQLAlchemy 2.0** ORM, implementando lógica de **idempotencia** para evitar registros duplicados.
4. **Verificación SQL:** Consultas `SELECT` analíticas para auditar la persistencia.

## 1. Instalación e Importación de Dependencias

In [ ]:
# Instalación de librerías requeridas (ideal para Google Colab o entorno nuevo)
!pip install kagglehub pandas sqlalchemy python-dotenv --quiet

import os
import shutil
from pathlib import Path
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine, text, String, Integer, Float, Boolean, DateTime
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

print("Librerías importadas correctamente.")

## 2. Configuración General y Rutas del Proyecto

In [ ]:
# Identificador del dataset en Kaggle
KAGGLE_DATASET = "srisyra02/air-quality-dataset"
CSV_FILENAME = "air_quality_dataset.csv"

# Creación de directorios para datos crudos (staging) y base de datos
os.makedirs("data/raw", exist_ok=True)
RAW_CSV_PATH = Path("data/raw") / CSV_FILENAME
DB_PATH = Path("data") / "air_quality.db"
DATABASE_URL = f"sqlite:///{DB_PATH.as_posix()}"

print(f"Dataset: {KAGGLE_DATASET}")
print(f"Archivo Staging: {RAW_CSV_PATH}")
print(f"Base de datos SQLite: {DATABASE_URL}")

## 3. Paso 1: Extracción (Extract) con kagglehub y Staging Area
Descargamos el dataset usando `kagglehub.dataset_download()` y guardamos una copia en `data/raw/` (patrón *Staging Area*). Si no hay conexión o ya existe una copia previa, el sistema la reutiliza como respaldo seguro.

In [ ]:
def extract_air_quality_data():
    print(f"[EXTRACT] Conectando con Kaggle para descargar: '{KAGGLE_DATASET}'...")
    try:
        import kagglehub
        download_path = kagglehub.dataset_download(KAGGLE_DATASET)
        download_dir = Path(download_path)
        csv_files = list(download_dir.glob("*.csv"))
        if not csv_files:
            raise FileNotFoundError("No se encontró archivo CSV en la carpeta descargada.")
        
        shutil.copy2(csv_files[0], RAW_CSV_PATH)
        print(f"[EXTRACT] Copia guardada en Staging Area: {RAW_CSV_PATH}")
    except Exception as error:
        print(f"[ALERTA] No se pudo descargar vía red: {error}")
        if RAW_CSV_PATH.exists():
            print(f"[EXTRACT] Utilizando respaldo local en Staging: {RAW_CSV_PATH}")
        else:
            raise error

    df_raw = pd.read_csv(RAW_CSV_PATH)
    print(f"[EXTRACT] Extracción exitosa: {len(df_raw)} filas y {len(df_raw.columns)} columnas.")
    return df_raw

df_raw = extract_air_quality_data()
df_raw.head()

## 4. Paso 2: Transformación (Transform) con Pandas
Aplicamos limpieza y enriquecimiento de datos:
1. **Normalización de nombres:** Renombrar a minúsculas (`pm25`, `pm10`, `nox`, `no2`, etc.).
2. **Identificador único:** Crear `measurement_id` para garantizar trazabilidad e idempotencia.
3. **Limpieza de valores físicos erróneos:** Sensores con valores negativos de gases se corrigen a 0.0.
4. **Feature Engineering:**
   - `quality_level`: Clasificación según PM2.5 (*Buena*, *Moderada*, *Dañina*).
   - `is_high_pollution`: Flag booleano (`True` si PM2.5 > 55.4 o CO2 > 500).
   - `ingested_at`: Fecha y hora de ingestión.
5. **Agregación analítica:** Tabla resumen agrupada por `quality_level` con promedios y máximos.

In [ ]:
def clasificar_calidad_aire(pm25_valor):
    if pm25_valor <= 35.4:
        return "Buena"
    elif pm25_valor <= 55.4:
        return "Moderada"
    else:
        return "Danina"

def transform_air_quality_data(df_raw):
    df = df_raw.copy()
    column_mapping = {
        "PM2.5": "pm25",
        "PM10": "pm10",
        "NOx": "nox",
        "NO2": "no2",
        "SO2": "so2",
        "VOCs": "vocs",
        "CO": "co",
        "CO2": "co2",
        "CH4": "ch4",
        "Temperature": "temperature"
    }
    df = df.rename(columns=column_mapping)
    df["measurement_id"] = range(1, len(df) + 1)

    # Limpieza de nulos
    df = df.dropna().copy()

    # Corrección de concentraciones negativas producidas por calibración de sensores
    columnas_gases = ["pm25", "pm10", "nox", "no2", "so2", "vocs", "co", "co2", "ch4"]
    for col in columnas_gases:
        df[col] = df[col].clip(lower=0.0)

    todas_num = columnas_gases + ["temperature"]
    df[todas_num] = df[todas_num].round(2)

    # Feature Engineering
    df["quality_level"] = df["pm25"].apply(clasificar_calidad_aire)
    df["is_high_pollution"] = (df["pm25"] > 55.4) | (df["co2"] > 500.0)
    now = datetime.now()
    df["ingested_at"] = now

    cols_measurements = [
        "measurement_id", "pm25", "pm10", "nox", "no2", "so2",
        "vocs", "co", "co2", "ch4", "temperature",
        "quality_level", "is_high_pollution", "ingested_at"
    ]
    df_measurements = df[cols_measurements].copy()

    # Resumen analítico agrupado
    df_summary = df.groupby("quality_level").agg(
        record_count=("measurement_id", "count"),
        avg_pm25=("pm25", "mean"),
        avg_pm10=("pm10", "mean"),
        avg_co2=("co2", "mean"),
        avg_temperature=("temperature", "mean"),
        max_pm25=("pm25", "max")
    ).reset_index()

    cols_agg = ["avg_pm25", "avg_pm10", "avg_co2", "avg_temperature", "max_pm25"]
    df_summary[cols_agg] = df_summary[cols_agg].round(2)
    df_summary["ingested_at"] = now

    return df_measurements, df_summary

df_measurements, df_summary = transform_air_quality_data(df_raw)
print(f"Filas transformadas: {len(df_measurements)}")
df_measurements.head()

### Vista de la tabla de resumen analítico generada:

In [ ]:
df_summary

## 5. Paso 3: Persistencia con SQLAlchemy 2.0 en SQLite
Definimos el esquema relacional con anotaciones de tipo (`Mapped` y `mapped_column`) según las mejores prácticas del Módulo 9.

In [ ]:
class Base(DeclarativeBase):
    pass

class AirQualityMeasurement(Base):
    __tablename__ = "air_quality_measurements"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    measurement_id: Mapped[int] = mapped_column(Integer, unique=True, nullable=False)
    pm25: Mapped[float] = mapped_column(Float, nullable=False)
    pm10: Mapped[float] = mapped_column(Float, nullable=False)
    nox: Mapped[float] = mapped_column(Float, nullable=False)
    no2: Mapped[float] = mapped_column(Float, nullable=False)
    so2: Mapped[float] = mapped_column(Float, nullable=False)
    vocs: Mapped[float] = mapped_column(Float, nullable=False)
    co: Mapped[float] = mapped_column(Float, nullable=False)
    co2: Mapped[float] = mapped_column(Float, nullable=False)
    ch4: Mapped[float] = mapped_column(Float, nullable=False)
    temperature: Mapped[float] = mapped_column(Float, nullable=False)
    quality_level: Mapped[str] = mapped_column(String(30), nullable=False)
    is_high_pollution: Mapped[bool] = mapped_column(Boolean, nullable=False)
    ingested_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)

class AirQualitySummary(Base):
    __tablename__ = "air_quality_summary"
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    quality_level: Mapped[str] = mapped_column(String(30), unique=True, nullable=False)
    record_count: Mapped[int] = mapped_column(Integer, nullable=False)
    avg_pm25: Mapped[float] = mapped_column(Float, nullable=False)
    avg_pm10: Mapped[float] = mapped_column(Float, nullable=False)
    avg_co2: Mapped[float] = mapped_column(Float, nullable=False)
    avg_temperature: Mapped[float] = mapped_column(Float, nullable=False)
    max_pm25: Mapped[float] = mapped_column(Float, nullable=False)
    ingested_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.now)

engine = create_engine(DATABASE_URL)
Base.metadata.create_all(engine)
print("Tablas SQLite creadas y verificadas exitosamente.")

## 6. Paso 4: Carga (Load) y Prevención de Duplicados (Idempotencia)
Implementamos la lógica para consultar previamente los registros ya existentes por `measurement_id`, garantizando que si el script se vuelve a ejecutar, no duplica información.

In [ ]:
def load_measurements(engine, df_measurements):
    with engine.connect() as conn:
        query = text("SELECT measurement_id FROM air_quality_measurements")
        filas = conn.execute(query).fetchall()
        ids_existentes = set(f[0] for f in filas)

    df_nuevos = df_measurements[~df_measurements["measurement_id"].isin(ids_existentes)].copy()
    if not df_nuevos.empty:
        df_nuevos.to_sql("air_quality_measurements", con=engine, if_exists="append", index=False)
        print(f"[LOAD] Insertados {len(df_nuevos)} nuevos registros.")
    else:
        print("[LOAD] Todos los registros ya existen en SQLite. Ninguna fila duplicada.")

def load_summary(engine, df_summary):
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM air_quality_summary"))
        df_summary.to_sql("air_quality_summary", con=conn, if_exists="append", index=False)
    print(f"[LOAD] Resumen analítico actualizado con {len(df_summary)} grupos.")

# Ejecución de Carga 1
print("--- Carga Inicial ---")
load_measurements(engine, df_measurements)
load_summary(engine, df_summary)

# Ejecución de Carga 2 (Comprobación de Idempotencia)
print("\n--- Segunda Ejecución (Prueba de Evitar Duplicados) ---")
load_measurements(engine, df_measurements)

## 7. Consultas SQL Analíticas de Verificación
Consultamos directamente la base de datos SQLite con sentencias `SELECT` para auditar la información almacenada.

In [ ]:
with engine.connect() as conn:
    print("1. Resumen por categoría de calidad del aire:")
    display(pd.read_sql_query("SELECT * FROM air_quality_summary ORDER BY avg_pm25 ASC", conn))

    print("\n2. Top 5 mediciones con mayor concentración de partículas PM2.5:")
    display(pd.read_sql_query("""
        SELECT measurement_id, pm25, pm10, co2, temperature, quality_level
        FROM air_quality_measurements
        ORDER BY pm25 DESC
        LIMIT 5;
    """, conn))

    print("\n3. Agregación según el flag de Alta Contaminación:")
    display(pd.read_sql_query("""
        SELECT 
            CASE WHEN is_high_pollution = 1 THEN 'Sí' ELSE 'No' END AS alta_contaminacion,
            COUNT(*) AS total_mediciones,
            ROUND(AVG(pm25), 2) AS pm25_promedio,
            ROUND(AVG(co2), 2) AS co2_promedio,
            ROUND(AVG(temperature), 2) AS temp_promedio
        FROM air_quality_measurements
        GROUP BY is_high_pollution;
    """, conn))

## 8. Conclusiones y Cumplimiento de la Pre-entrega
- **Conexión a fuente:** Dataset de Kaggle (`srisyra02/air-quality-dataset`) integrado mediante `kagglehub` con respaldo en Staging Area.
- **Transformación y Limpieza:** Nombres normalizados, valores negativos corregidos, features calculadas y tabla resumen agregada.
- **Capa de Persistencia:** Esquema relacional implementado en SQLite con SQLAlchemy 2.0 y archivo DDL `sql/create_tables.sql`.
- **Idempotencia:** Algoritmo que previene registros duplicados en ejecuciones sucesivas.